In [1]:
# ============================================================
# 01_data_load_keystroke_mouse.ipynb
# Adaptive Continuous Authentication — Data Loader
# ============================================================

"""
Goal:
  1. Load CMU Keystroke Dynamics (DSL-StrongPasswordData.csv)
  2. Load Balabit Mouse Dynamics Challenge (train/test splits)
  3. Normalize and unify into one parquet dataset for later feature extraction

Output:
  data/unified_auth_stream.parquet
"""
# ============================================================
# Imports
# ============================================================
import os
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

# ============================================================
# Paths
# ============================================================
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

KEYSTROKE_PATH = DATA_DIR / "DSL-StrongPasswordData.csv"
MOUSE_DIR = DATA_DIR / "Mouse-Dynamics-Challenge"

In [2]:
# ============================================================
# SECTION 1 — Load CMU Keystroke (DSL-StrongPasswordData)
# ============================================================

def load_cmu_keystroke(csv_path: Path) -> pd.DataFrame:
    """
    Load the DSL-StrongPasswordData.csv file.
    Each row corresponds to a typing repetition of a fixed password.

    Columns include dwell (H.*) and latency (DD.*, UD.*) features.
    """
    if not csv_path.exists():
        raise FileNotFoundError(f"❌ File not found: {csv_path}")
    
    df = pd.read_csv(csv_path)
    df.rename(columns={"subject": "user_id", "sessionIndex": "session_id"}, inplace=True)
    df["modality"] = "keystroke"
    df["timestamp"] = df["rep"]  # proxy timestamp
    
    print(f"✅ Loaded keystroke data: {len(df)} samples, {df['user_id'].nunique()} users")
    print(f"Feature columns: {len(df.columns) - 4} (excluding id/session/rep/modality)")
    return df


# ============================================================
# SECTION 2 — Load Balabit Mouse Dynamics
# ============================================================

def parse_balabit_mouse(base_dir: Path) -> pd.DataFrame:
    """
    Parse the Balabit Mouse Dynamics dataset organized as:
      {train_files,test_files}/user_{id}/session_{sessionId}
    Each session file is a CSV (no extension).
    """
    records = []

    if not base_dir.exists():
        raise FileNotFoundError(f"❌ Missing Balabit directory: {base_dir}")

    for split in ["training_files", "test_files", "train_files"]:  # support both naming styles
        split_dir = base_dir / split
        if not split_dir.exists():
            continue

        for user_dir in sorted(split_dir.glob("user*")):
            user_id = user_dir.name.replace("user", "").replace("_", "")
            for session_file in sorted(user_dir.iterdir()):
                if not session_file.is_file():
                    continue
                session_id = session_file.name
                try:
                    df = pd.read_csv(session_file)
                except Exception as e:
                    print(f"⚠️ Skipping {session_file}: {e}")
                    continue

                expected_cols = ["record timestamp", "client timestamp", "button", "state", "x", "y"]
                for col in expected_cols:
                    if col not in df.columns:
                        df[col] = np.nan

                df["user_id"] = user_id
                df["session_id"] = session_id
                df["split"] = split
                records.append(df)

    if not records:
        raise RuntimeError(f"❌ No Balabit files found in {base_dir}")

    mouse_df = pd.concat(records, ignore_index=True)
    print(f"✅ Loaded Balabit data: {len(mouse_df)} rows from {mouse_df['user_id'].nunique()} users")
    return mouse_df


# ============================================================
# SECTION 3 — Unify Schema
# ============================================================

def unify_modalities(df_keys: pd.DataFrame, df_mouse: pd.DataFrame) -> pd.DataFrame:
    """
    Convert both datasets to a unified schema:
      ['user_id','session_id','timestamp','modality','features']
    """

    # Keystroke: flatten dwell & latency features into np.array
    key_feature_cols = [
        c for c in df_keys.columns
        if c not in ["user_id", "session_id", "rep", "modality", "timestamp"]
    ]
    ks = df_keys[["user_id", "session_id", "timestamp", "modality"] + key_feature_cols].copy()
    ks["features"] = ks[key_feature_cols].apply(lambda row: row.values.astype(np.float32), axis=1)
    ks = ks[["user_id", "session_id", "timestamp", "modality", "features"]]

    # Mouse: use raw positions as basic features for now
    ms = df_mouse[["user_id", "session_id", "client timestamp", "x", "y"]].copy()
    ms.rename(columns={"client timestamp": "timestamp"}, inplace=True)
    ms["modality"] = "mouse"
    ms["features"] = ms[["x", "y"]].apply(lambda r: np.array(r, dtype=np.float32), axis=1)
    ms = ms[["user_id", "session_id", "timestamp", "modality", "features"]]

    unified = pd.concat([ks, ms], ignore_index=True)
    unified.sort_values(["user_id", "timestamp"], inplace=True)
    unified.reset_index(drop=True, inplace=True)

    # --- 🔧 FIX: enforce consistent string typing for IDs ---
    unified["user_id"] = unified["user_id"].astype(str)
    unified["session_id"] = unified["session_id"].astype(str)

    print(f"✅ Unified dataset shape: {unified.shape}")
    return unified


# ============================================================
# SECTION 4 — Save Unified Dataset
# ============================================================

def save_unified(df: pd.DataFrame):
    out_path = DATA_DIR / "unified_auth_stream.parquet"
    df.to_parquet(out_path, index=False)
    print(f"💾 Saved unified dataset → {out_path.resolve()}")

In [3]:
if __name__ == "__main__":
    print("🚀 Loading CMU Keystroke + Balabit Mouse Dynamics...")

    df_keys = load_cmu_keystroke(KEYSTROKE_PATH)
    df_mouse = parse_balabit_mouse(MOUSE_DIR)

    df_unified = unify_modalities(df_keys, df_mouse)
    save_unified(df_unified)

    print("✅ Data preparation complete.")

🚀 Loading CMU Keystroke + Balabit Mouse Dynamics...
✅ Loaded keystroke data: 20400 samples, 51 users
Feature columns: 32 (excluding id/session/rep/modality)
✅ Loaded Balabit data: 4609929 rows from 10 users
✅ Unified dataset shape: (4630329, 5)
💾 Saved unified dataset → /Users/vkhawarey/Documents/git/moviecruncher/dataviz/adaptive_auth/data/unified_auth_stream.parquet
✅ Data preparation complete.
